In [ ]:
import yfinance as yf
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"


headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

tables = pd.read_html(response.text)
sp500_table = tables[0]

tickers = sp500_table['Symbol'].tolist()

tickers = tickers[:10]

print(f"Downloading {len(tickers)} stocks: {tickers}")

data = yf.download(tickers, start="2020-01-01", end="2024-01-01")

data.head()

In [ ]:
import time

def download_single_stock(ticker, start, end):

    try:
        data = yf.download(ticker, start=start, end=end, progress=False)
        return ticker, data
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")
        return ticker, None

def download_all_single_process(tickers, start, end):

    results = {}

    start_time = time.time()

    for ticker in tickers:
        print(f"Downloading {ticker}...")
        ticker_name, data = download_single_stock(ticker, start, end)
        results[ticker_name] = data

    end_time = time.time()
    elapsed = end_time - start_time

    return results, elapsed


start_date = "2023-01-01"
end_date = "2024-01-01"

results_single, time_single = download_all_single_process(tickers, start_date, end_date)
print(f"\n Single Process Time: {time_single:.2f} seconds")

In [ ]:
from multiprocessing import Pool
import multiprocessing as mp

def download_stock_wrapper(args):
    ticker, start, end = args
    return download_single_stock(ticker, start, end)

def download_all_multiprocessing(tickers, start, end, n_processes=4):

    start_time = time.time()

    args = [(ticker, start, end) for ticker in tickers]

    with Pool(processes=n_processes) as pool:
        results_list = pool.map(download_stock_wrapper, args)

    results = {ticker: data for ticker, data in results_list}

    end_time = time.time()
    elapsed = end_time - start_time

    return results, elapsed


results_multi, time_multi = download_all_multiprocessing(tickers, start_date, end_date, n_processes=4)
print(f"\n Multiprocessing Time: {time_multi:.2f} seconds")
print(f" Speedup: {time_single/time_multi:.2f}x faster!")

In [ ]:
import pickle

with open('sp500_data_single.pkl', 'wb') as f:
    pickle.dump(results_single, f)

with open('sp500_data_multi.pkl', 'wb') as f:
    pickle.dump(results_multi, f)

print(" Data saved!")

In [ ]:
import matplotlib.pyplot as plt

methods = ['Single Process', 'Multiprocessing']
times = [time_single, time_multi]
colors = ['red', 'green']

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, times, color=colors, alpha=0.7, edgecolor='black')

for bar, time_val in zip(bars, times):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{time_val:.2f}s',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylabel('Time (seconds)', fontsize=12)
plt.title(f'Download Time Comparison ({len(tickers)} stocks, 1 year data)',
          fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

speedup = time_single / time_multi
plt.text(0.5, max(times) * 0.9,
         f'Speedup: {speedup:.2f}x faster',
         ha='center', fontsize=14,
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.savefig('multiprocessing_comparison.png', dpi=300)
plt.show()

print(f" Visualization saved!")

Task 2:

In [ ]:
ls = [1, 2, 3, 4, 5, 6, 7, 8, 9]

squared_loop = []
for num in ls:
    squared_loop.append(num ** 2)
print(f"Squared (loop): {squared_loop}")

squared_map = list(map(lambda x: x**2, ls))
print(f"Squared (map): {squared_map}")

def is_prime(n):

    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

primes_loop = []
for num in squared_map:
    if is_prime(num):
        primes_loop.append(num)
print(f"Primes (loop): {primes_loop}")

primes_filter = list(filter(is_prime, squared_map))
print(f"Primes (filter): {primes_filter}")

sum_loop = 0
for num in primes_filter:
    sum_loop += num
print(f"Sum (loop): {sum_loop}")

from functools import reduce

sum_reduce = reduce(lambda acc, x: acc + x, primes_filter, 0)
print(f"Sum (reduce): {sum_reduce}")

sum_builtin = sum(primes_filter)
print(f"Sum (built-in): {sum_builtin}")

mean_value = sum(ls) / len(ls)

mse_loop = 0
for x in ls:
    mse_loop += (x - mean_value) ** 2
mse_loop = mse_loop / len(ls)
print(f"MSE (loop): {mse_loop:.4f}")

mse_functional = reduce(
    lambda acc, x: acc + (x - mean_value)**2,
    ls,
    0
) / len(ls)
print(f"MSE (functional): {mse_functional:.4f}")

In [ ]:
import numpy as np
from statistics import mean, variance

ls_np = np.array(ls)

squared_np = ls_np ** 2
print(f"Squared (numpy): {squared_np}")

is_prime_vec = np.vectorize(is_prime)
primes_np = squared_np[is_prime_vec(squared_np)]
print(f"Primes (numpy): {primes_np}")

sum_np = np.sum(primes_np)
print(f"Sum (numpy): {sum_np}")

mse_np = np.mean((ls_np - np.mean(ls_np))**2)
print(f"MSE (numpy): {mse_np:.4f}")

mse_var = np.var(ls_np)  # variance = MSE
print(f"MSE = Variance (numpy): {mse_var:.4f}")

mse_stats = variance(ls)
print(f"MSE (statistics): {mse_stats:.4f}")

In [ ]:
import pandas as pd
import numpy as np

# Sample DataFrame
df = pd.DataFrame({
    'A': [1, 2, 3, 4, 5],
    'B': [10, 20, 30, 40, 50],
    'C': [100, 200, 300, 400, 500]
})

print("Original DataFrame:")
print(df)

df['A_squared'] = df['A'].apply(lambda x: x**2)
print("\nAfter squaring A:")
print(df)

df['B_filtered'] = df['B'].apply(lambda x: x if x > 25 else 0)
print("\nAfter filtering B:")
print(df)

# A + B
df['A_plus_B'] = df.apply(lambda row: row['A'] + row['B'], axis=1)
print("\nA + B:")
print(df)

def categorize(row):

    if row['A'] > 3 and row['B'] > 30:
        return 'High'
    elif row['A'] > 2:
        return 'Medium'
    else:
        return 'Low'

df['Category'] = df.apply(categorize, axis=1)
print("\nWith categories:")
print(df)

df['C_div_100'] = df['C'].map(lambda x: x / 100)
print("\nC divided by 100:")
print(df)